
### Dataset Processing / Normalization

This Jupyter Notebook processes multiple datasets to prepare them for machine learning tasks. Below is a summary of the workflow:

1. **Datasets**:
    - A list of datasets (`datasets`) is defined, each containing the file path, the protected attribute (e.g., `age`, `sex`), and the separator used in the file.
    - The source of these datasets will be detailed in the thesis.

2. **Data Preprocessing**:
    - Each dataset is read using `pd.read_csv()`.
    - Missing values are filled with `0`, and duplicate rows are removed.
        - We do not remove missing values as it reduces drastically the volume of data, for datasets like COMPAS it return 0 rows.
    - Non-numeric columns are encoded using either:
      - **One-Hot Encoding**: For columns with fewer than 10 unique values.
      - **Label Encoding**: For columns with more than 10 unique values.
    - Protected attributes are skipped during encoding.

3. **Conversion Mapping**:
    - A dictionary (`d_conversions`) is created to store the encoding mappings for each column. This includes:
      - One-hot encoded column names.
      - Label encoding mappings.

4. **Saving Results**:
    - The processed dataset is saved as a CSV file.
    - The encoding mappings are saved as a JSON file for reference.

This code ensures that categorical variables are properly encoded for machine learning models while preserving the integrity of protected attributes. It also documents the transformations applied to the data for reproducibility.

In [1]:
import pandas as pd
import json

In [2]:
datasets = [("datasets/bank-full.csv", ['marital', 'y'], ";"),
            ("datasets/compas-scores-raw.csv", ["race", "is_recid"], ","),
            ("datasets/dropout.csv", ['Gender', 'Target'], ";"),
            ("datasets/Employee.csv", ['Gender', 'LeaveOrNot'], ","),
            ("datasets/indian.csv", ['Sex', 'Target'], ","),
            ("datasets/adult.data", ['sex', 'income'], ","),]

In [3]:
for dataset, attrs, sep in datasets:
    skipped_protected = False
    skipped_target = False
    protected_attr = attrs[0]
    target_attr = attrs[1]
    df = pd.read_csv(dataset, sep=sep)
    df.fillna(0, inplace=True)
    df.drop_duplicates(inplace=True)
    d_conversions = {}
    
    for column in df.columns:
        if column == protected_attr:
            # check if protected attribute is binary
            if df[column].nunique() != 2:
                raise ValueError(f"Protected attribute '{column}' is not binary in dataset '{dataset}'")
            print(f"Skipping protected attribute '{column}' for dataset '{dataset}'")
            skipped_protected = True
            continue
        if column == target_attr:
            skipped_target = True
            print(f"Skipping target attribute '{column}' for dataset '{dataset}'")
            continue
            
        # Check if column is numeric
        if pd.api.types.is_numeric_dtype(df[column]):
            print(f"Column '{column}' is already numeric, skipping...")
            continue
            
        # For non-numeric columns, decide between label encoding or one-hot encoding
        unique_values = df[column].nunique()
        
        if unique_values <= 10:  # One-hot encode if few categories
            print(f"One-hot encoding '{column}' with {unique_values} categories")
            # Create dummy variables
            dummies = pd.get_dummies(df[column], prefix=column, drop_first=False)
            # Drop original column and add dummy columns
            df = df.drop(column, axis=1)
            df = pd.concat([df, dummies], axis=1)
            # Save the mapping for reference
            d_conversions[column] = f"one_hot_encoded_to_{list(dummies.columns)}"
        else:  # Label encode if many categories
            print(f"Label encoding '{column}' with {unique_values} categories")
            df[column] = df[column].astype('category')
            mapping = dict(zip(df[column].cat.categories, range(len(df[column].cat.categories))))
            d_conversions[column] = mapping
            df[column] = df[column].cat.codes
    if not skipped_protected:
        raise ValueError(f"Protected attribute '{protected_attr}' missing in dataset '{dataset}'")
    if not skipped_target:
        raise ValueError(f"Target attribute '{target_attr}' missing in dataset '{dataset}'")
    # Save conversions and dataset

    dataset_name = dataset.split('/')[-1].split('.')[0]
    with open(f'datasets/{dataset_name}_conversions.json', 'w') as f:
        json.dump(d_conversions, f, indent=4)
    
    df.to_csv(f'datasets/{dataset_name}_converted.csv', index=False)

Column 'age' is already numeric, skipping...
Label encoding 'job' with 12 categories


ValueError: Protected attribute 'marital' is not binary in dataset 'datasets/bank-full.csv'

In [5]:
bank_df = pd.read_csv("datasets/bank-full.csv", sep=";")
bank_df.marital.value_counts()

married     27214
single      12790
divorced     5207
Name: marital, dtype: int64